# Финальная модель

Самодостаточный pipeline: метрики, генерация признаков, target encoding, временная валидация, holdout и test-предсказания.

Для фиксации Python hash seed запускай Jupyter так: `PYTHONHASHSEED=42 jupyter notebook final_model.ipynb`. Остальные генераторы случайности инициализируются в первой ячейке кода.


In [1]:
from __future__ import annotations

import os
import random
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold, TimeSeriesSplit
from xgboost import XGBClassifier

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

DATA_DIR = Path("data")
OUTPUT_DIR = Path("results")
MODEL_PATH = OUTPUT_DIR / "final_model.json"
SUBMISSION_PATH = OUTPUT_DIR / "submission.csv"
ITEM_SMOOTHING = 1.0

MODEL_PARAMS = {
    "n_estimators": 750,
    "learning_rate": 0.03,
    "max_depth": 3,
    "min_child_weight": 5,
    "subsample": 0.85,
    "colsample_bytree": 0.85,
    "reg_lambda": 5.0,
    "reg_alpha": 0.1,
    "gamma": 0.0,
    "scale_pos_weight": 8,
    "objective": "binary:logistic",
    "eval_metric": "logloss",
    "tree_method": "hist",
    "random_state": SEED,
    "n_jobs": -1,
}

EVENT_TYPES = [
    "search_results_view",
    "item_view",
    "photo_swipe",
    "seller_page_view",
    "contact_phone_show",
    "contact_chat_open",
    "contact_message_sent",
    "favorite_add",
    "login",
    "captcha_shown",
]

ENCODING_FIELDS = {
    "item_id": 1.0,
    "search_query": 10.0,
    "user_agent": 15.0,
    "item_category": 20.0,
    "item_location": 20.0,
    "seller_type": 20.0,
    "platform": 15.0,
    "event_name": 20.0,
}


## Метрики


In [2]:
TARGET_RECALL = 0.70

def pr_curve(y_true, score) -> tuple[np.ndarray, np.ndarray]:
    """(precision, recall) в точках на границах групп одинакового score."""
    y_true = np.asarray(y_true, dtype=int)
    score = np.asarray(score, dtype=float)
    if y_true.shape != score.shape:
        raise ValueError("y_true и score разной длины")

    n_pos = int(y_true.sum())
    if n_pos == 0:
        return np.array([]), np.array([])

    order = np.argsort(-score, kind="mergesort")
    y, s = y_true[order], score[order]

    tp = np.cumsum(y)
    k = np.arange(1, len(y) + 1)
    ends = np.r_[s[1:] != s[:-1], True]
    return tp[ends] / k[ends], tp[ends] / n_pos

def precision_at_recall(y_true, score, recall: float = TARGET_RECALL) -> float:
    """Максимальный precision среди порогов с recall >= `recall`."""
    prec, rec = pr_curve(y_true, score)
    if len(prec) == 0:
        return float("nan")
    ok = rec >= recall
    return float(prec[ok].max()) if ok.any() else 0.0

def recall_at_fpr(y_true, score, fpr: float = 0.01) -> float:
    """Посчитать долю ботов при ограничении на ложные срабатывания."""
    y_true = np.asarray(y_true, dtype=int)
    score = np.asarray(score, dtype=float)
    n_pos, n_neg = int(y_true.sum()), int((1 - y_true).sum())
    if n_pos == 0 or n_neg == 0:
        return float("nan")

    order = np.argsort(-score, kind="mergesort")
    y, s = y_true[order], score[order]
    ends = np.r_[s[1:] != s[:-1], True]
    tp = np.cumsum(y)[ends]
    fp = np.cumsum(1 - y)[ends]

    allowed = fp <= fpr * n_neg
    return float(tp[allowed].max() / n_pos) if allowed.any() else 0.0


def evaluate_scores(y_true, score) -> dict[str, float]:
    """Собрать основные метрики классификации."""
    return {
        "precision_at_recall_70": precision_at_recall(y_true, score, TARGET_RECALL),
        "average_precision": average_precision_score(y_true, score),
        "roc_auc": roc_auc_score(y_true, score),
        "recall_at_fpr_1pct": recall_at_fpr(y_true, score),
    }


## Базовая модель

Базовая модель использует возраст cookie, объём активности, разнообразие действий и счётчики типов событий. Она обучается тем же XGBoost с теми же параметрами, что и расширенное решение. Обе модели сравниваются на одинаковых временных фолдах и holdout. Итоговая таблица находится в конце ноутбука.


In [3]:
train = pd.read_csv(
    DATA_DIR / "train.csv",
    parse_dates=["cookie_created_at", "window_start_ts", "window_end_ts"],
)
test = pd.read_csv(
    DATA_DIR / "test.csv",
    parse_dates=["cookie_created_at", "window_start_ts", "window_end_ts"],
)
events = pd.read_csv(DATA_DIR / "events.csv", parse_dates=["event_ts"])

dates = train.window_start_ts.dt.normalize()
cv_mask = dates.between("2026-04-06", "2026-04-16")
holdout_mask = dates.between("2026-04-17", "2026-04-19")
cv_dates = np.sort(dates.loc[cv_mask].unique())

def build_baseline_features(metadata: pd.DataFrame, events: pd.DataFrame) -> pd.DataFrame:
    """Собрать простые признаки по событиям и возрасту cookie."""
    meta = metadata[["cookie_id", "cookie_created_at", "window_start_ts", "window_end_ts"]].copy()
    for column in ["cookie_created_at", "window_start_ts", "window_end_ts"]:
        meta[column] = pd.to_datetime(meta[column])

    ev = events.merge(
        meta[["cookie_id", "window_start_ts", "window_end_ts"]],
        on="cookie_id",
        how="inner",
        validate="many_to_one",
    )
    ev = ev.loc[
        ev.event_ts.ge(ev.window_start_ts) & ev.event_ts.lt(ev.window_end_ts)
    ].copy()
    if ev.empty:
        raise ValueError("No events fall inside the supplied observation windows")

    grouped = ev.groupby("cookie_id", sort=False, observed=True)
    features = grouped.agg(
        n_events=("event_ts", "size"),
        n_event_types=("event_name", "nunique"),
        n_items=("item_id", "nunique"),
        n_categories=("item_category", "nunique"),
        n_queries=("search_query", "nunique"),
    )
    event_counts = pd.crosstab(ev.cookie_id, ev.event_name).reindex(
        columns=EVENT_TYPES, fill_value=0
    )
    event_counts.columns = [f"event_count__{name}" for name in event_counts.columns]
    features = features.join(event_counts)

    meta_indexed = meta.set_index("cookie_id")
    features["cookie_age_days"] = (
        meta_indexed.window_start_ts - meta_indexed.cookie_created_at
    ).dt.total_seconds() / 86400
    return features.reindex(meta.cookie_id).fillna(0).reset_index(drop=True)

baseline_features = build_baseline_features(train, events)
y = train.target.to_numpy(dtype=int)
baseline_fold_metrics = []
baseline_splitter = TimeSeriesSplit(n_splits=5, test_size=1)

for fold, (fit_date_pos, valid_date_pos) in enumerate(
    baseline_splitter.split(cv_dates), start=1
):
    fit_dates = cv_dates[fit_date_pos]
    valid_dates = cv_dates[valid_date_pos]
    fit_mask = dates.isin(fit_dates)
    valid_mask = dates.isin(valid_dates)
    fit_pos = np.flatnonzero(fit_mask.to_numpy())
    valid_pos = np.flatnonzero(valid_mask.to_numpy())

    model = XGBClassifier(**MODEL_PARAMS)
    model.fit(baseline_features.iloc[fit_pos], y[fit_pos])
    score = model.predict_proba(baseline_features.iloc[valid_pos])[:, 1]
    baseline_fold_metrics.append({
        "fold": fold,
        "train_end": pd.Timestamp(fit_dates[-1]).date().isoformat(),
        "validation_date": pd.Timestamp(valid_dates[0]).date().isoformat(),
        "fit_rows": len(fit_pos),
        "validation_rows": len(valid_pos),
        **evaluate_scores(y[valid_pos], score),
    })

baseline_cv_results = pd.DataFrame(baseline_fold_metrics)
baseline_cv_positions = np.flatnonzero(cv_mask.to_numpy())
baseline_holdout_positions = np.flatnonzero(holdout_mask.to_numpy())
baseline_model = XGBClassifier(**MODEL_PARAMS)
baseline_model.fit(baseline_features.iloc[baseline_cv_positions], y[baseline_cv_positions])
baseline_holdout_score = baseline_model.predict_proba(
    baseline_features.iloc[baseline_holdout_positions]
)[:, 1]
baseline_holdout_metrics = evaluate_scores(
    y[baseline_holdout_positions], baseline_holdout_score
)

display(pd.DataFrame({
    "period": ["5-fold CV: 6–16 Apr", "holdout: 17–19 Apr", "test"],
    "rows": [int(cv_mask.sum()), int(holdout_mask.sum()), len(test)],
}))
display(baseline_cv_results)
display(pd.DataFrame([baseline_holdout_metrics], index=["baseline holdout"]))


,period,rows
0,5-fold CV: 6–16 Apr,9140
1,holdout: 17–19 Apr,1951
2,test,4909


,fold,train_end,validation_date,fit_rows,validation_rows,precision_at_recall_70,average_precision,roc_auc,recall_at_fpr_1pct
0,1,2026-04-11,2026-04-12,5113,817,0.262570,0.563218,0.847622,0.313433
1,2,2026-04-12,2026-04-13,5930,843,0.230303,0.467250,0.841820,0.320755
2,3,2026-04-13,2026-04-14,6773,826,0.314103,0.561720,0.864267,0.371429
3,4,2026-04-14,2026-04-15,7599,801,0.282209,0.529784,0.849394,0.369231
4,5,2026-04-15,2026-04-16,8400,740,0.215385,0.454219,0.819314,0.283333


,precision_at_recall_70,average_precision,roc_auc,recall_at_fpr_1pct
baseline holdout,0.267303,0.475396,0.830248,0.24375


## Окна событий и нормализация категорий


In [4]:
def _windowed_events(metadata: pd.DataFrame, events: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Оставить события внутри окна каждого cookie."""
    meta = metadata.assign(
        window_start_ts=pd.to_datetime(metadata["window_start_ts"]),
        window_end_ts=pd.to_datetime(metadata["window_end_ts"]),
    )
    ev = events.copy()
    ev["event_ts"] = pd.to_datetime(ev["event_ts"])
    ev = ev.merge(
        meta[["cookie_id", "window_start_ts", "window_end_ts"]],
        on="cookie_id",
        how="inner",
        validate="many_to_one",
    )
    in_window = (
        ev["event_ts"].ge(ev["window_start_ts"])
        & ev["event_ts"].lt(ev["window_end_ts"])
    )
    return meta, ev.loc[in_window].copy()

def _norm_platform(value: object) -> str:
    """Нормализовать название платформы."""
    if pd.isna(value):
        return "unknown"
    value = str(value).strip().lower()
    return {"web": "web", "android": "android", "ios": "ios", "iphone": "ios"}.get(value, value or "unknown")

def _browser_family(value: object) -> str:
    """Определить браузер по строке User Agent."""
    text = "" if pd.isna(value) else str(value).lower()
    if "headlesschrome" in text:
        return "headless_chrome"
    if "yabrowser" in text:
        return "yandex"
    if "edg/" in text or "edge/" in text:
        return "edge"
    if "opr/" in text or "opera" in text:
        return "opera"
    if "firefox/" in text:
        return "firefox"
    if "chrome/" in text or "chromium" in text:
        return "chrome"
    if "safari/" in text:
        return "safari"
    if "avito/" in text or "okhttp" in text:
        return "avito_app"
    return "other"

def _os_family(value: object) -> str:
    """Определить ОС по строке User Agent."""
    text = "" if pd.isna(value) else str(value).lower()
    if "android" in text:
        return "android"
    if "iphone" in text or "ipad" in text or "ios" in text:
        return "ios"
    if "windows" in text:
        return "windows"
    if "mac os" in text or "macintosh" in text:
        return "macos"
    if "linux" in text or "x11" in text:
        return "linux"
    return "other"

def _mode(series: pd.Series) -> str:
    """Вернуть самое частое значение, а при равенстве выбрать первое по алфавиту."""
    values = series.dropna().astype(str)
    if values.empty:
        return "unknown"
    counts = values.value_counts()
    return sorted(counts[counts == counts.max()].index)[0]


## Признаки событий и поведения


In [5]:
def _prepare_event_features(events: pd.DataFrame) -> pd.DataFrame:
    """Подготовить поля событий для агрегации."""
    ev = events
    ev["event_name"] = ev["event_name"].fillna("unknown").astype(str)
    ev["platform_norm"] = ev["platform"].map(_norm_platform)
    ev["browser_family"] = ev["user_agent"].map(_browser_family)
    ev["os_family"] = ev["user_agent"].map(_os_family)
    ev["ua_length"] = ev["user_agent"].fillna("").astype(str).str.len()
    ua_lower = ev["user_agent"].fillna("").astype(str).str.lower()
    ev["ua_headless"] = ua_lower.str.contains("headless", regex=False).astype("int8")
    ev["ua_app"] = (ua_lower.str.contains("avito/", regex=False) | ua_lower.str.contains("okhttp", regex=False)).astype("int8")
    ev["ua_mobile"] = (ua_lower.str.contains("android", regex=False) | ua_lower.str.contains("iphone", regex=False) | ua_lower.str.contains("ipad", regex=False)).astype("int8")

    elapsed = (ev["event_ts"] - ev["window_start_ts"]).dt.total_seconds()
    ev["elapsed_seconds"] = elapsed
    ev["minute_bin"] = np.floor(elapsed / 60).astype("int16")
    ev["hour_bin"] = np.floor(elapsed / 3600).astype("int8")
    ev["clock_hour"] = ev["event_ts"].dt.hour.astype("int8")
    ev["is_night"] = ev["clock_hour"].between(0, 5).astype("int8")
    ev["search_query_length"] = ev["search_query"].fillna("").astype(str).str.len().where(ev["search_query"].notna())
    return ev

def _aggregate_event_features(events: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Посчитать основные числовые и категориальные агрегаты."""
    grouped = events.groupby("cookie_id", sort=False, observed=True)
    features = grouped.agg(
        n_events=("event_ts", "size"),
        n_event_types=("event_name", "nunique"),
        n_items=("item_id", "nunique"),
        n_categories=("item_category", "nunique"),
        n_locations=("item_location", "nunique"),
        n_seller_types=("seller_type", "nunique"),
        n_queries=("search_query", "nunique"),
        n_platforms=("platform_norm", "nunique"),
        n_browsers=("browser_family", "nunique"),
        n_os_families=("os_family", "nunique"),
        search_page_count=("search_page", "count"),
        search_page_mean=("search_page", "mean"),
        search_page_max=("search_page", "max"),
        search_query_length_mean=("search_query_length", "mean"),
        search_query_length_max=("search_query_length", "max"),
        ua_length_mean=("ua_length", "mean"),
        ua_headless_share=("ua_headless", "mean"),
        ua_app_share=("ua_app", "mean"),
        ua_mobile_share=("ua_mobile", "mean"),
        first_elapsed_seconds=("elapsed_seconds", "min"),
        last_elapsed_seconds=("elapsed_seconds", "max"),
        active_minutes=("minute_bin", "nunique"),
        active_hours=("hour_bin", "nunique"),
        night_event_count=("is_night", "sum"),
    )
    categorical = grouped.agg(
        platform_mode=("platform_norm", _mode),
        browser_mode=("browser_family", _mode),
        os_mode=("os_family", _mode),
    )
    return features, categorical

def _add_event_mix(features: pd.DataFrame, events: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Добавить счётчики и доли типов событий."""
    event_counts = pd.crosstab(events["cookie_id"], events["event_name"]).reindex(columns=EVENT_TYPES, fill_value=0)
    event_counts = event_counts.rename(columns={name: f"event_{name}_count" for name in EVENT_TYPES})
    event_shares = event_counts.div(features["n_events"], axis=0)
    event_shares.columns = [name.replace("_count", "_share") for name in event_counts.columns]
    features = features.join(event_counts, how="left").join(event_shares, how="left")
    return features, event_shares

def _add_gap_features(features: pd.DataFrame, events: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Посчитать интервалы между событиями."""
    ordered = events.sort_values(["cookie_id", "event_ts"], kind="mergesort").copy()
    ordered["gap_seconds"] = ordered.groupby("cookie_id", sort=False)["event_ts"].diff().dt.total_seconds()
    gaps = ordered.groupby("cookie_id", sort=False)["gap_seconds"]
    gap_stats = gaps.agg(
        gap_mean_seconds="mean",
        gap_median_seconds="median",
        gap_std_seconds="std",
        gap_min_seconds="min",
        gap_max_seconds="max",
    )
    gap_features = pd.DataFrame(index=gap_stats.index)
    gap_features["gap_cv"] = gap_stats["gap_std_seconds"] / gap_stats["gap_mean_seconds"].replace(0, np.nan)
    gap_features["zero_gap_share"] = gaps.apply(lambda values: values.eq(0).mean())
    gap_features["gap_under_1s_share"] = gaps.apply(lambda values: values.lt(1).mean())
    gap_features["gap_under_5s_share"] = gaps.apply(lambda values: values.lt(5).mean())
    gap_features["gap_under_30s_share"] = gaps.apply(lambda values: values.lt(30).mean())
    return features.join(gap_stats).join(gap_features), ordered

def _add_activity_features(features: pd.DataFrame, metadata: pd.DataFrame) -> pd.DataFrame:
    """Посчитать длительность активности cookie."""
    features["active_span_seconds"] = features["last_elapsed_seconds"] - features["first_elapsed_seconds"]
    features["first_event_delay_seconds"] = features["first_elapsed_seconds"]
    window_duration_seconds = (
        metadata["window_end_ts"] - metadata["window_start_ts"]
    ).dt.total_seconds()
    features["last_event_remaining_seconds"] = window_duration_seconds - features["last_elapsed_seconds"]
    features["events_per_active_hour"] = features["n_events"] / features["active_hours"].clip(lower=1)
    features["night_event_share"] = features["night_event_count"] / features["n_events"]
    return features

def _add_cookie_age_feature(features: pd.DataFrame, metadata: pd.DataFrame) -> pd.DataFrame:
    """Посчитать возраст cookie на начало окна."""
    features["cookie_age_days"] = (
        metadata["window_start_ts"] - metadata["cookie_created_at"]
    ).dt.total_seconds() / 86400
    return features

def _add_transition_features(features: pd.DataFrame, ordered_events: pd.DataFrame) -> pd.DataFrame:
    """Посчитать смены типа события."""
    ordered_events["previous_event_name"] = ordered_events.groupby("cookie_id", sort=False)["event_name"].shift()
    ordered_events["event_type_changed"] = (
        ordered_events["previous_event_name"].notna()
        & ordered_events["event_name"].ne(ordered_events["previous_event_name"])
    )
    transitions = ordered_events.groupby("cookie_id", sort=False)["event_type_changed"].agg(["sum", "mean"])
    transitions.columns = ["event_type_transitions", "event_type_change_share"]
    return features.join(transitions)

def _add_distribution_features(
    features: pd.DataFrame,
    events: pd.DataFrame,
    event_shares: pd.DataFrame,
) -> pd.DataFrame:
    """Посчитать энтропию и концентрацию активности."""
    by_hour = events.groupby(["cookie_id", "clock_hour"], observed=True).size().unstack(fill_value=0)
    hour_p = by_hour.div(by_hour.sum(axis=1), axis=0)
    features["hour_entropy"] = -(hour_p.where(hour_p > 0) * np.log(hour_p.where(hour_p > 0))).sum(axis=1)
    features["peak_hour_share"] = hour_p.max(axis=1)
    event_p = event_shares.clip(lower=1e-12)
    features["event_mix_entropy"] = -(event_p * np.log(event_p)).sum(axis=1)
    return features

def _restore_cookie_rows(
    features: pd.DataFrame,
    categorical: pd.DataFrame,
    metadata: pd.DataFrame,
) -> pd.DataFrame:
    """Вернуть все cookie в исходном порядке."""
    features = features.reindex(metadata["cookie_id"]).fillna(0)
    categorical = categorical.reindex(metadata["cookie_id"]).fillna("unknown")
    features = features.join(categorical)
    features.index.name = "cookie_id"
    result = features.reset_index()
    if result["cookie_id"].tolist() != metadata["cookie_id"].tolist():
        raise AssertionError("Feature order or row count changed during aggregation")
    return result

def build_features(metadata: pd.DataFrame, events: pd.DataFrame) -> pd.DataFrame:
    """Собрать временные и поведенческие признаки по cookie."""
    meta, windowed_events = _windowed_events(metadata, events)
    meta["cookie_created_at"] = pd.to_datetime(meta["cookie_created_at"])
    metadata_features = meta.set_index("cookie_id")
    if windowed_events.empty:
        raise ValueError("No events fall inside the supplied observation windows")

    prepared_events = _prepare_event_features(windowed_events)
    features, categorical = _aggregate_event_features(prepared_events)
    features, event_shares = _add_event_mix(features, prepared_events)
    features, ordered_events = _add_gap_features(features, prepared_events)
    features = _add_activity_features(features, metadata_features)
    features = _add_transition_features(features, ordered_events)
    features = _add_distribution_features(features, prepared_events, event_shares)
    features = _add_cookie_age_feature(features, metadata_features)
    return _restore_cookie_rows(features, categorical, meta)


## Категориальные и курсорные признаки


In [6]:
def encode_features(train_features: pd.DataFrame, test_features: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Закодировать категории платформы, браузера и ОС."""
    categorical_columns = ["platform_mode", "browser_mode", "os_mode"]
    train_x = train_features.drop(columns=["cookie_id"]).copy()
    test_x = test_features.drop(columns=["cookie_id"]).copy()
    combined = pd.concat([train_x, test_x], ignore_index=True)
    combined = pd.get_dummies(combined, columns=categorical_columns, dtype=np.uint8)
    train_encoded = combined.iloc[: len(train_x)].copy()
    test_encoded = combined.iloc[len(train_x) :].copy()
    return train_encoded, test_encoded

def _pointer_position_features(points: pd.DataFrame) -> pd.DataFrame:
    """Сводно описать координаты курсора."""
    grouped = points.groupby("cookie_id", sort=False)
    stats = grouped.agg(
        pointer_points=("event_ts", "size"),
        pointer_unique_x=("pointer_x", "nunique"),
        pointer_unique_y=("pointer_y", "nunique"),
        pointer_x_mean=("pointer_x", "mean"),
        pointer_y_mean=("pointer_y", "mean"),
        pointer_x_std=("pointer_x", "std"),
        pointer_y_std=("pointer_y", "std"),
        pointer_x_min=("pointer_x", "min"),
        pointer_x_max=("pointer_x", "max"),
        pointer_y_min=("pointer_y", "min"),
        pointer_y_max=("pointer_y", "max"),
        pointer_x_q10=("pointer_x", lambda x: x.quantile(.10)),
        pointer_x_q50=("pointer_x", "median"),
        pointer_x_q90=("pointer_x", lambda x: x.quantile(.90)),
        pointer_y_q10=("pointer_y", lambda x: x.quantile(.10)),
        pointer_y_q50=("pointer_y", "median"),
        pointer_y_q90=("pointer_y", lambda x: x.quantile(.90)),
    )
    stats["pointer_xy_unique"] = grouped.apply(
        lambda frame: frame[["pointer_x", "pointer_y"]].drop_duplicates().shape[0]
    )
    stats["pointer_x_range"] = stats.pointer_x_max - stats.pointer_x_min
    stats["pointer_y_range"] = stats.pointer_y_max - stats.pointer_y_min
    stats["pointer_x_std_norm"] = stats.pointer_x_std / stats.pointer_x_range.replace(0, np.nan)
    stats["pointer_y_std_norm"] = stats.pointer_y_std / stats.pointer_y_range.replace(0, np.nan)
    stats["pointer_x_q50_norm"] = (stats.pointer_x_q50 - stats.pointer_x_min) / stats.pointer_x_range.replace(0, np.nan)
    stats["pointer_y_q50_norm"] = (stats.pointer_y_q50 - stats.pointer_y_min) / stats.pointer_y_range.replace(0, np.nan)
    return stats

def _pointer_path_features(points: pd.DataFrame) -> pd.DataFrame:
    """Посчитать скорость, направление и длину пути."""
    points = points.copy()
    grouped = points.groupby("cookie_id", sort=False)
    points["prev_x"] = grouped.pointer_x.shift()
    points["prev_y"] = grouped.pointer_y.shift()
    points["prev_ts"] = grouped.event_ts.shift()
    points["dx"] = points.pointer_x - points.prev_x
    points["dy"] = points.pointer_y - points.prev_y
    points["dt_seconds"] = (points.event_ts - points.prev_ts).dt.total_seconds()
    points["distance"] = np.sqrt(points.dx.pow(2) + points.dy.pow(2))
    points["speed"] = points.distance / points.dt_seconds.where(points.dt_seconds > 0)
    points["direction"] = (np.arctan2(points.dy, points.dx) + 2 * np.pi) % (2 * np.pi)
    points["direction_bin"] = np.floor(points.direction / (2 * np.pi / 8)).clip(0, 7)
    stats = points.groupby("cookie_id").agg(
        pointer_path_length=("distance", "sum"),
        pointer_speed_mean=("speed", "mean"),
        pointer_speed_median=("speed", "median"),
        pointer_speed_q90=("speed", lambda x: x.quantile(.90)),
        pointer_dt_median=("dt_seconds", "median"),
        pointer_zero_move_share=("distance", lambda x: x.eq(0).mean()),
        pointer_move_under_5_share=("distance", lambda x: x.lt(5).mean()),
        pointer_move_over_50_share=("distance", lambda x: x.gt(50).mean()),
        pointer_direction_nunique=("direction_bin", "nunique"),
    )
    directions = pd.crosstab(points.cookie_id, points.direction_bin).reindex(columns=range(8), fill_value=0)
    direction_p = directions.div(directions.sum(axis=1).replace(0, 1), axis=0)
    stats["pointer_direction_entropy"] = -(
        direction_p.where(direction_p > 0) * np.log(direction_p.where(direction_p > 0))
    ).sum(axis=1)
    first_last = points.groupby("cookie_id").agg(
        first_x=("pointer_x", "first"), first_y=("pointer_y", "first"),
        last_x=("pointer_x", "last"), last_y=("pointer_y", "last"),
    )
    stats["pointer_net_distance"] = np.sqrt(
        (first_last.last_x - first_last.first_x).pow(2)
        + (first_last.last_y - first_last.first_y).pow(2)
    )
    stats["pointer_straightness"] = (
        stats.pointer_net_distance / stats.pointer_path_length.replace(0, np.nan)
    )
    return stats

def _pointer_event_coverage(points: pd.DataFrame, events: pd.DataFrame) -> pd.DataFrame:
    """Посчитать долю событий с координатами курсора."""
    coverage = pd.crosstab(points.cookie_id, points.event_name).reindex(
        columns=sorted(events.event_name.dropna().unique()), fill_value=0
    )
    totals = pd.crosstab(events.cookie_id, events.event_name).reindex(
        columns=coverage.columns, fill_value=0
    )
    shares = coverage.div(totals.replace(0, np.nan)).fillna(0)
    shares.columns = [f"pointer_share_by_event__{name}" for name in shares.columns]
    return shares

def _empty_pointer_features(metadata: pd.DataFrame) -> pd.DataFrame:
    """Вернуть нулевую долю курсорных событий."""
    result = pd.DataFrame(index=pd.Index(metadata.cookie_id, name="cookie_id"))
    result["pointer_event_share"] = 0.0
    return result.reset_index()

def pointer_features(metadata: pd.DataFrame, events: pd.DataFrame) -> pd.DataFrame:
    """Собрать признаки движения курсора."""
    _, windowed_events = _windowed_events(metadata, events)
    result = pd.DataFrame(index=pd.Index(metadata.cookie_id, name="cookie_id"))
    all_counts = windowed_events.groupby("cookie_id").size().rename("n_events")
    points = windowed_events.dropna(subset=["pointer_x", "pointer_y"]).copy()
    points = points.sort_values(["cookie_id", "event_ts", "event_name"], kind="mergesort")
    if points.empty:
        return _empty_pointer_features(metadata)

    points["pointer_x"] = pd.to_numeric(points.pointer_x, errors="coerce")
    points["pointer_y"] = pd.to_numeric(points.pointer_y, errors="coerce")
    points = points.dropna(subset=["pointer_x", "pointer_y"])
    position_stats = _pointer_position_features(points)
    path_stats = _pointer_path_features(points)
    result = result.join(position_stats).join(path_stats)
    result["pointer_event_share"] = result.pointer_points / all_counts
    result = result.join(_pointer_event_coverage(points, windowed_events))
    result.index.name = "cookie_id"
    return result.reindex(metadata.cookie_id).fillna(0).reset_index()


## Target encoding со сглаживанием


In [7]:
def make_value_tables(events: pd.DataFrame, metadata: pd.DataFrame) -> dict[str, pd.DataFrame]:
    """Посчитать частоты категорий по cookie."""
    _, ev = _windowed_events(metadata, events)

    tables = {}
    for field in ENCODING_FIELDS:
        selected = ev.loc[ev[field].notna(), ["cookie_id", field]].copy()
        selected[field] = selected[field].astype(str)
        counts = selected.groupby(["cookie_id", field], sort=False).size()
        counts = counts.rename("event_count").reset_index()
        tables[field] = counts.rename(columns={field: "value"})
    return tables

def target_stats(
    value_table: pd.DataFrame,
    labels: pd.Series,
    source_ids: list[str],
    prior: float,
    smoothing: float,
) -> pd.DataFrame:
    """Сгладить долю класса к общей доле."""
    source = value_table.loc[value_table.cookie_id.isin(source_ids)].merge(
        labels.rename("target"),
        left_on="cookie_id",
        right_index=True,
        how="inner",
        validate="many_to_one",
    )
    grouped = source.groupby("value").target.agg(
        n_cookies="size",
        positive_cookies="sum",
    )
    grouped["rate"] = (
        grouped.positive_cookies + smoothing * prior
    ) / (grouped.n_cookies + smoothing)
    return grouped[["n_cookies", "rate"]]

def _prepare_encoded_values(
    value_table: pd.DataFrame,
    stats: pd.DataFrame,
    prior: float,
    cookie_ids: list[str],
) -> pd.DataFrame:
    """Подготовить значения и веса событий."""
    selected = value_table.loc[value_table.cookie_id.isin(cookie_ids)].copy()
    selected = selected.join(stats, on="value")
    selected["known"] = selected.rate.notna()
    selected["rate"] = selected.rate.fillna(prior)
    selected["log_support"] = np.log1p(selected.n_cookies.fillna(0))
    selected["weighted_rate_sum"] = selected.rate * selected.event_count
    selected["known_events"] = selected.event_count * selected.known.astype(int)
    selected["log_support_sum"] = selected.log_support * selected.event_count
    return selected

def _aggregate_encoded_values(selected: pd.DataFrame) -> pd.DataFrame:
    """Свернуть категориальные значения по cookie."""
    agg = selected.groupby("cookie_id").agg(
        weighted_rate_sum=("weighted_rate_sum", "sum"),
        event_count=("event_count", "sum"),
        max_rate=("rate", "max"),
        rate_std=("rate", "std"),
        known_events=("known_events", "sum"),
        log_support_sum=("log_support_sum", "sum"),
    )
    return agg

def _format_cookie_encoding(
    agg: pd.DataFrame,
    prior: float,
    cookie_ids: list[str],
    field_name: str,
) -> pd.DataFrame:
    """Собрать итоговые признаки целевой кодировки."""
    result = pd.DataFrame(index=pd.Index(cookie_ids, name="cookie_id"))
    result[f"te__{field_name}__weighted_rate"] = (
        agg.weighted_rate_sum.div(agg.event_count).reindex(result.index).fillna(prior)
    )
    result[f"te__{field_name}__max_rate"] = agg.max_rate.reindex(result.index).fillna(prior)
    result[f"te__{field_name}__rate_std"] = agg.rate_std.reindex(result.index).fillna(0)
    result[f"te__{field_name}__known_event_share"] = (
        agg.known_events.div(agg.event_count).reindex(result.index).fillna(0)
    )
    result[f"te__{field_name}__mean_log_support"] = (
        agg.log_support_sum.div(agg.event_count).reindex(result.index).fillna(0)
    )
    return result

def encode_cookie_values(
    value_table: pd.DataFrame,
    stats: pd.DataFrame,
    prior: float,
    cookie_ids: list[str],
    field_name: str,
) -> pd.DataFrame:
    """Собрать целевые признаки категорий для cookie."""
    selected = _prepare_encoded_values(value_table, stats, prior, cookie_ids)
    agg = _aggregate_encoded_values(selected)
    return _format_cookie_encoding(agg, prior, cookie_ids, field_name)

def _encode_from_source(
    value_table: pd.DataFrame,
    labels: pd.Series,
    source_ids: list[str],
    target_ids: list[str],
    smoothing: float,
    field_name: str,
) -> pd.DataFrame:
    """Закодировать cookie по меткам источника."""
    prior = float(labels.loc[source_ids].mean())
    mapping = target_stats(value_table, labels, source_ids, prior, smoothing)
    return encode_cookie_values(value_table, mapping, prior, target_ids, field_name)

def _crossfit_training_field(
    value_table: pd.DataFrame,
    labels: pd.Series,
    fit_ids: list[str],
    field_name: str,
    smoothing: float,
    folds: StratifiedKFold,
) -> pd.DataFrame:
    """Закодировать train без его собственных меток."""
    fit_field = pd.DataFrame(index=pd.Index(fit_ids, name="cookie_id"))
    for train_pos, held_pos in folds.split(np.zeros(len(fit_ids)), labels.to_numpy()):
        source_ids = [fit_ids[i] for i in train_pos]
        held_ids = [fit_ids[i] for i in held_pos]
        encoded = _encode_from_source(
            value_table, labels, source_ids, held_ids, smoothing, field_name
        )
        if fit_field.empty:
            fit_field = pd.DataFrame(index=fit_field.index, columns=encoded.columns, dtype=float)
        fit_field.loc[held_ids, encoded.columns] = encoded.loc[held_ids].to_numpy()
    return fit_field

def build_encoding_for_smoothing(
    value_tables: dict[str, pd.DataFrame],
    labels: pd.Series,
    fit_ids: list[str],
    valid_ids: list[str],
    item_smoothing: float = 1.0,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Построить сглажированную кодировку без утечки меток."""
    fit_labels = labels.loc[fit_ids]
    folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    fit_result = pd.DataFrame(index=pd.Index(fit_ids, name="cookie_id"))
    valid_result = pd.DataFrame(index=pd.Index(valid_ids, name="cookie_id"))

    for field, default_smoothing in ENCODING_FIELDS.items():
        smoothing = item_smoothing if field == "item_id" else default_smoothing
        value_table = value_tables[field]
        fit_field = _crossfit_training_field(
            value_table, fit_labels, fit_ids, field, smoothing, folds
        )
        fit_result = fit_result.join(fit_field)
        valid_field = _encode_from_source(
            value_table, fit_labels, fit_ids, valid_ids, smoothing, field
        )
        valid_result = valid_result.join(valid_field)

    return fit_result.fillna(0), valid_result.fillna(0)


## Подготовка матриц признаков


In [8]:
metadata = pd.concat([train.drop(columns="target"), test], ignore_index=True)
features = build_features(metadata, events)
train_features = features.iloc[: len(train)].reset_index(drop=True)
test_features = features.iloc[len(train) :].reset_index(drop=True)
X_base_train, X_base_test = encode_features(train_features, test_features)

pointer = pointer_features(metadata, events).drop(columns="cookie_id")
value_tables = make_value_tables(events, metadata)
labels = train.set_index("cookie_id").target

cv_ids = train.loc[cv_mask, "cookie_id"].tolist()
holdout_ids = train.loc[holdout_mask, "cookie_id"].tolist()


## Временная кросс-валидация


In [9]:
fold_metrics = []
folds = TimeSeriesSplit(n_splits=5, test_size=1)

for fold, (fit_date_pos, valid_date_pos) in enumerate(folds.split(cv_dates), start=1):
    fit_dates = cv_dates[fit_date_pos]
    valid_dates = cv_dates[valid_date_pos]
    fit_mask = dates.isin(fit_dates)
    valid_mask = dates.isin(valid_dates)
    fit_ids = train.loc[fit_mask, "cookie_id"].tolist()
    valid_ids = train.loc[valid_mask, "cookie_id"].tolist()
    fit_pos = np.flatnonzero(fit_mask.to_numpy())
    valid_pos = np.flatnonzero(valid_mask.to_numpy())

    fit_te, valid_te = build_encoding_for_smoothing(
        value_tables=value_tables,
        labels=labels,
        fit_ids=fit_ids,
        valid_ids=valid_ids,
        item_smoothing=ITEM_SMOOTHING,
    )
    X_fit = pd.concat([
        X_base_train.iloc[fit_pos].set_axis(fit_ids, axis=0),
        fit_te,
        pointer.iloc[fit_pos].set_axis(fit_ids, axis=0),
        ], axis=1).fillna(0)
    X_valid = pd.concat([
        X_base_train.iloc[valid_pos].set_axis(valid_ids, axis=0),
        valid_te,
        pointer.iloc[valid_pos].set_axis(valid_ids, axis=0),
        ], axis=1).fillna(0)
    y_fit = labels.loc[fit_ids].to_numpy(dtype=int)
    y_valid = labels.loc[valid_ids].to_numpy(dtype=int)

    model = XGBClassifier(**MODEL_PARAMS)
    model.fit(X_fit, y_fit)
    score = model.predict_proba(X_valid)[:, 1]
    fold_metrics.append({
        "fold": fold,
        "train_end": pd.Timestamp(fit_dates[-1]).date().isoformat(),
        "validation_date": pd.Timestamp(valid_dates[0]).date().isoformat(),
        "fit_rows": len(y_fit),
        "validation_rows": len(y_valid),
        **evaluate_scores(y_valid, score),
    })

cv_results = pd.DataFrame(fold_metrics)
display(cv_results)
display(cv_results[[
    "precision_at_recall_70", "average_precision", "roc_auc", "recall_at_fpr_1pct"
]].mean().to_frame("mean across folds").T)


,fold,train_end,validation_date,fit_rows,validation_rows,precision_at_recall_70,average_precision,roc_auc,recall_at_fpr_1pct
0,1,2026-04-11,2026-04-12,5113,817,0.626667,0.778823,0.929592,0.626866
1,2,2026-04-12,2026-04-13,5930,843,0.904762,0.797169,0.927776,0.792453
2,3,2026-04-13,2026-04-14,6773,826,0.830508,0.819846,0.953269,0.671429
3,4,2026-04-14,2026-04-15,7599,801,0.920000,0.817457,0.920778,0.723077
4,5,2026-04-15,2026-04-16,8400,740,0.877551,0.794437,0.956691,0.716667


,precision_at_recall_70,average_precision,roc_auc,recall_at_fpr_1pct
mean across folds,0.831898,0.801546,0.937621,0.706098


## Переобучение и holdout


In [10]:
cv_pos = np.flatnonzero(cv_mask.to_numpy())
holdout_pos = np.flatnonzero(holdout_mask.to_numpy())
train_validation_te, holdout_te = build_encoding_for_smoothing(
    value_tables=value_tables,
    labels=labels,
    fit_ids=cv_ids,
    valid_ids=holdout_ids,
    item_smoothing=ITEM_SMOOTHING,
)
X_train_validation = pd.concat([
    X_base_train.iloc[cv_pos].set_axis(cv_ids, axis=0),
    train_validation_te,
    pointer.iloc[cv_pos].set_axis(cv_ids, axis=0),
    ], axis=1).fillna(0)
X_holdout = pd.concat([
    X_base_train.iloc[holdout_pos].set_axis(holdout_ids, axis=0),
    holdout_te,
    pointer.iloc[holdout_pos].set_axis(holdout_ids, axis=0),
    ], axis=1).fillna(0)
y_train_validation = labels.loc[cv_ids].to_numpy(dtype=int)
y_holdout = labels.loc[holdout_ids].to_numpy(dtype=int)

final_model = XGBClassifier(**MODEL_PARAMS)
final_model.fit(X_train_validation, y_train_validation)
holdout_score = final_model.predict_proba(X_holdout)[:, 1]
holdout_metrics = pd.DataFrame([{
    "fit_rows": len(y_train_validation),
    "holdout_rows": len(y_holdout),
    **evaluate_scores(y_holdout, holdout_score),
}])
display(holdout_metrics)


,fit_rows,holdout_rows,precision_at_recall_70,average_precision,roc_auc,recall_at_fpr_1pct
0,9140,1951,0.88189,0.784224,0.941283,0.7


## Предсказания для test


In [11]:
test_ids = test.cookie_id.tolist()
test_pos = np.arange(len(train), len(train) + len(test))
_, test_te = build_encoding_for_smoothing(
    value_tables=value_tables,
    labels=labels,
    fit_ids=cv_ids,
    valid_ids=test_ids,
    item_smoothing=ITEM_SMOOTHING,
)
X_test = pd.concat([
    X_base_test.set_axis(test_ids, axis=0),
    test_te,
    pointer.iloc[test_pos].set_axis(test_ids, axis=0),
], axis=1).fillna(0)
test_score = final_model.predict_proba(X_test)[:, 1]
submission = pd.DataFrame({"cookie_id": test_ids, "score": test_score})
display(submission.head())
print("Test rows:", len(submission))


,cookie_id,score
0,ck_315fb710a0e371e7,0.001477
1,ck_a76ee3b3e3e522fd,0.352814
2,ck_94c9a4d382689e82,0.153963
3,ck_8eaf9509ad9462a0,0.002082
4,ck_9a88a5a989cb5bc6,0.500526


Test rows: 4909


## Сохранение модели и submission


In [12]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
final_model.save_model(str(MODEL_PATH))
submission.to_csv(SUBMISSION_PATH, index=False)
print("Saved model:", MODEL_PATH)
print("Saved submission:", SUBMISSION_PATH)


Saved model: results/final_model.json
Saved submission: results/submission.csv


## Сравнение моделей

Средние метрики кросс-валидации и метрики одного временного holdout рассчитаны для одинаковых дат и целевых меток. Положительная разница означает преимущество расширенной модели.


In [1]:
comparison_metrics = [
    "precision_at_recall_70",
    "average_precision",
    "roc_auc",
    "recall_at_fpr_1pct",
]
baseline_cv_mean = baseline_cv_results[comparison_metrics].mean()
enhanced_cv_mean = cv_results[comparison_metrics].mean()
baseline_holdout_values = pd.Series(baseline_holdout_metrics)
enhanced_holdout_values = holdout_metrics.loc[0, comparison_metrics]

comparison = pd.DataFrame({
    "CV baseline": baseline_cv_mean,
    "CV enhanced": enhanced_cv_mean,
    "CV difference": enhanced_cv_mean - baseline_cv_mean,
    "Holdout baseline": baseline_holdout_values[comparison_metrics],
    "Holdout enhanced": enhanced_holdout_values,
    "Holdout difference": enhanced_holdout_values - baseline_holdout_values[comparison_metrics],
})
comparison.index.name = "metric"
display(comparison)


NameError: name 'baseline_cv_results' is not defined